In [2]:
import pandas as pd
from langchain_huggingface import HuggingFacePipeline
from langchain import PromptTemplate, LLMChain
from langchain.memory import ConversationBufferMemory
import torch

In [ ]:
# 1. Configurar o modelo do Hugging Face usando langchain-huggingface
model_name =  "meta-llama/Llama-3.2-3B-Instruct" #"meta-llama/Llama-3.3-70B-Instruct" # "dice-research/lola_v1_alpaca_instructions_multilingual" # "ricdomolm/lawma-8b" # 

llm = HuggingFacePipeline.from_model_id(
    model_id=model_name,         # Modelo desejado
    task="text-generation",      # Tarefa de geração de texto
    pipeline_kwargs={"max_new_tokens": 2024}
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ValueError: Cannot set a non-string value as the pad_token

In [ ]:
# 2. Ler o arquivo .parquet com as decisões judiciais
df = pd.read_parquet("validation.parquet")
df_test = df[0:2]

In [ ]:
# 3. Ler o prompt detalhado de um arquivo .txt (instruções para extração)
with open("prompt/teste.txt", "r", encoding="utf-8") as f:
    extraction_instructions = f.read()


In [ ]:
# 4. Configurar a memória de conversa e definir um prompt inicial do sistema
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
system_message = (
    "Você é um assistente jurídico especialista em extrair informações de sentenças judiciais. "
    "Extraia cuidadosamente os dados relevantes do documento e retorne a extração final em formato JSON. "
    "Utilize as instruções a seguir para guiar sua resposta:\n\n"
    + extraction_instructions
)
# Salva a mensagem do sistema no histórico
memory.save_context({"input": system_message}, {"output": ""})


In [ ]:
# 5. Configurar o PromptTemplate para o chat
# O template inclui o histórico da conversa e a nova entrada
chat_prompt = PromptTemplate(
    template="{chat_history}\nUsuário: {input}\nAssistente:",
    input_variables=["chat_history", "input"]
)

# 6. Criar a chain do chat com memória
chat_chain = LLMChain(llm=llm, prompt=chat_prompt, memory=memory)


In [ ]:
# 7. Função de chunking para dividir o texto em partes menores
def chunk_text(text, max_words=900):
    words = text.split()
    chunks = []
    current_chunk = []
    current_count = 0
    for word in words:
        current_chunk.append(word)
        current_count += 1
        if current_count >= max_words:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_count = 0
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks


In [ ]:
# 8. Processar cada decisão judicial usando a abordagem de chat incremental
results = []
for idx, row in df_test.iterrows():
    decision_text = row["julgado"]
    print(f"Processing decision {idx} with {len(decision_text.split())} words")
    
    # Dividir o texto da sentença em chunks
    chunks = chunk_text(decision_text, max_words=1000)
    
    # Para cada chunk, enviar como entrada para o chat e atualizar o histórico
    for chunk in chunks:
        # Chame a chain; o método invoke() espera um dicionário com a chave "input"
        assistant_response = chat_chain.invoke(input={"input": chunk})
        print(f"Chunk processed, assistant response:\n{assistant_response}\n{'-'*60}\n")
    
    # Após enviar todos os chunks, a última mensagem do assistente no histórico será a extração final
    # Aqui, pegamos a última mensagem armazenada na memória
    if memory.buffer:
        final_extraction = memory.buffer[-1].content
    else:
        final_extraction = ""
    results.append(final_extraction)
    print(f"Decision {idx} final extraction:\n{final_extraction}\n{'='*60}\n")
    
    # Opcional: limpar a memória para o próximo documento (se desejar que cada decisão tenha um contexto separado)
    memory.clear()


In [ ]:
results[0]